# 02. Infinite Well via Dirichlet QST Split Evolution

The initial state is a Gaussian multiplied by \(\sin(\pi x/L)\), so it satisfies the hard-wall Dirichlet boundary condition. A standard periodic QFT models a ring rather than a box; this workflow uses the boundary-compatible quantum-sine-transform (QST) family, implemented numerically as an orthonormal DST-II. Because the ideal infinite well has zero potential inside the box, changing the step count mainly changes circuit repetition cost rather than the physics error.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
COMMON_NOTEBOOK = str(PROJECT_ROOT / "notebooks" / "00_common_qft_split_functions.ipynb")
%run "$COMMON_NOTEBOOK"

FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

In [2]:
L = 10.0
N = 64
hbar = 1.0
m = 1.0

x0 = L / 2.0
k0 = 2.0
sigma = 0.8

t_max = 6.0
r = 100
snapshot_count = 5

reference_grid_size = 4097
reference_tail_tolerance = 1e-10
reference_basis_cap_min = 192

fidelity_sweep_r = [20, 40, 60, 80, 100]
spatial_convergence_N = [32, 64, 128]

In [3]:
result = run_infinite_well_case(
    grid_size=N,
    length=L,
    x0=x0,
    k0=k0,
    sigma=sigma,
    t_max=t_max,
    steps=r,
    mass=m,
    hbar=hbar,
    reference_grid_size=reference_grid_size,
    reference_tail_tolerance=reference_tail_tolerance,
    reference_basis_cap=max(reference_basis_cap_min, 4 * N),
)
summary = fidelity_summary(result)

display(Markdown("## Main Run Summary"))
display(pd.DataFrame([summary]))
display(
    Markdown(
        f"Reference basis kept **{result['reference_data']['n_keep']}** sine states out of cap "
        f"**{result['reference_data']['basis_cap']}**, with discarded tail estimate "
        f"**{result['reference_data']['tail_weight']:.2e}**."
    )
)

## Main Run Summary

,minimum_fidelity,final_time_fidelity,mean_fidelity,max_split_norm_error,max_reference_norm_error,max_edge_probability
0,1.0,1.0,1.0,3.441691e-14,4.440892e-16,0.242073


Reference basis kept **20** sine states out of cap **256**, with discarded tail estimate **4.47e-12**.

In [4]:
parameter_df = pd.DataFrame(
    [
        {
            "system": "infinite_well",
            "numerical_transform": "orthonormal_DST-II_quantum_sine_transform_family",
            "boundary_model": "Dirichlet_hard_wall",
            "periodic_boundary_mismatch": False,
            "N": N,
            "n_qubits": int(np.log2(N)),
            "domain": f"(0, {L}) midpoint grid",
            "L": L,
            "x0": x0,
            "k0": k0,
            "sigma": sigma,
            "t_max": t_max,
            "r": r,
            "dt": result["dt"],
            "hbar": hbar,
            "m": m,
            "reference_grid_size": result["reference_data"]["dense_grid_size"],
            "reference_basis_cap": result["reference_data"]["basis_cap"],
            "reference_basis_kept": result["reference_data"]["n_keep"],
            "reference_tail_weight": result["reference_data"]["tail_weight"],
            "initial_state_modifier": "sin(pi x / L)",
            "boundary_caveat": "resolved_by_boundary_compatible_QST_not_periodic_ring_QFT",
            "trotter_note": "pure_kinetic_well_so_step_count_changes_circuit_cost_not_physics_error",
            "fidelity_sweep_r": ",".join(str(value) for value in fidelity_sweep_r),
            "spatial_convergence_N": ",".join(str(value) for value in spatial_convergence_N),
        }
    ]
)
fidelity_df = pd.DataFrame(
    {
        "time": result["times"],
        "fidelity": result["fidelity"],
        "split_norm": result["split_norms"],
        "reference_norm": result["reference_norms"],
    }
)

save_dataframe(parameter_df, TABLES_DIR, "infinite_well_parameters.csv")
save_dataframe(fidelity_df, TABLES_DIR, "infinite_well_fidelity_vs_time.csv")

display(Markdown("## Simulation Parameters"))
display(parameter_df)

## Simulation Parameters

,system,numerical_transform,boundary_model,periodic_boundary_mismatch,N,n_qubits,domain,L,x0,k0,...,m,reference_grid_size,reference_basis_cap,reference_basis_kept,reference_tail_weight,initial_state_modifier,boundary_caveat,trotter_note,fidelity_sweep_r,spatial_convergence_N
0,infinite_well,orthonormal_DST-II_quantum_sine_transform_family,Dirichlet_hard_wall,False,64,6,"(0, 10.0) midpoint grid",10.0,5.0,2.0,...,1.0,4097,256,20,4.473977e-12,sin(pi x / L),resolved_by_boundary_compatible_QST_not_period...,pure_kinetic_well_so_step_count_changes_circui...,"20,40,60,80,100","32,64,128"


In [5]:
snapshot_figure = plot_density_snapshots(
    x=result["x"],
    times=result["times"],
    reference_states=result["reference_states"],
    split_states=result["split_states"],
    snapshot_count=snapshot_count,
    title="Infinite-well probability density",
)
fidelity_figure = plot_fidelity(result["times"], result["fidelity"], "Infinite-well fidelity")

for stale_path in list(FIGURES_DIR.glob("infinite_well_density_snapshots_page_*.png")) + list(FIGURES_DIR.glob("infinite_well_density_snapshots_page_*.pdf")):
    stale_path.unlink()

save_publication_figure(snapshot_figure, FIGURES_DIR, "infinite_well_density_snapshots")
save_publication_figure(fidelity_figure, FIGURES_DIR, "infinite_well_fidelity_vs_time")
plt.close(snapshot_figure)
plt.close(fidelity_figure)

print("Saved infinite-well density and fidelity figures.")

Saved infinite-well density and fidelity figures.


In [6]:
sweep_records = []
for r_value in fidelity_sweep_r:
    local = run_infinite_well_case(
        grid_size=N,
        length=L,
        x0=x0,
        k0=k0,
        sigma=sigma,
        t_max=t_max,
        steps=int(r_value),
        mass=m,
        hbar=hbar,
        reference_grid_size=reference_grid_size,
        reference_tail_tolerance=reference_tail_tolerance,
        reference_basis_cap=max(reference_basis_cap_min, 4 * N),
    )
    local_summary = fidelity_summary(local)
    sweep_records.append(
        {
            "system": "infinite_well",
            "r": int(r_value),
            "dt": local["dt"],
            "final_time_fidelity": local_summary["final_time_fidelity"],
            "minimum_fidelity": local_summary["minimum_fidelity"],
            "mean_fidelity": local_summary["mean_fidelity"],
        }
    )

fidelity_sweep_df = pd.DataFrame(sweep_records)
save_dataframe(fidelity_sweep_df, TABLES_DIR, "infinite_well_fidelity_vs_r.csv")
display(Markdown("## Trotter-Step Convergence"))
display(fidelity_sweep_df)

## Trotter-Step Convergence

,system,r,dt,final_time_fidelity,minimum_fidelity,mean_fidelity
0,infinite_well,20,0.300,1.0,1.0,1.0
1,infinite_well,40,0.150,1.0,1.0,1.0
2,infinite_well,60,0.100,1.0,1.0,1.0
3,infinite_well,80,0.075,1.0,1.0,1.0
4,infinite_well,100,0.060,1.0,1.0,1.0


In [7]:
convergence_records = []
for n_value in spatial_convergence_N:
    local = run_infinite_well_case(
        grid_size=int(n_value),
        length=L,
        x0=x0,
        k0=k0,
        sigma=sigma,
        t_max=t_max,
        steps=r,
        mass=m,
        hbar=hbar,
        reference_grid_size=reference_grid_size,
        reference_tail_tolerance=reference_tail_tolerance,
        reference_basis_cap=max(reference_basis_cap_min, 4 * int(n_value)),
    )
    local_summary = fidelity_summary(local)
    convergence_records.append(
        {
            "system": "infinite_well",
            "N": int(n_value),
            "n_qubits": int(np.log2(n_value)),
            "dx": local["dx"],
            "final_time_fidelity": local_summary["final_time_fidelity"],
            "minimum_fidelity": local_summary["minimum_fidelity"],
        }
    )

spatial_convergence_df = pd.DataFrame(convergence_records)
save_dataframe(spatial_convergence_df, TABLES_DIR, "infinite_well_spatial_convergence.csv")
convergence_figure = plot_convergence(
    spatial_convergence_df,
    x_column="N",
    y_column="final_time_fidelity",
    title="Infinite-well spatial convergence",
    xlabel="Grid points, N",
)
save_publication_figure(convergence_figure, FIGURES_DIR, "infinite_well_spatial_convergence")
plt.close(convergence_figure)

display(Markdown("## Spatial Convergence"))
display(spatial_convergence_df)

## Spatial Convergence

,system,N,n_qubits,dx,final_time_fidelity,minimum_fidelity
0,infinite_well,32,5,0.312500,1.0,1.0
1,infinite_well,64,6,0.156250,1.0,1.0
2,infinite_well,128,7,0.078125,1.0,1.0
